In [ ]:
# 1. 환경 설정 (Drive 마운트 + 패키지 설치 + 캐시를 Drive로)
import os, subprocess, sys, shutil

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")
CACHE_DIR  = os.path.join(DRIVE_DIR, ".cache")

# Resume ckpt는 로컬에 저장 (빠르고 디스크 절약, 세션 종료 시 사라짐)
LOCAL_CKPT = "/content/dacon/local_ckpt"

for d in [
    DRIVE_DIR,
    os.path.join(DRIVE_DIR, "checkpoints"),
    os.path.join(DRIVE_DIR, "submissions"),
    CACHE_DIR,
    WORK_DIR,
    LOCAL_CKPT,
]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_DIR, "huggingface")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_DIR, "torch")
os.environ["XDG_CACHE_HOME"] = CACHE_DIR
os.environ["PIP_CACHE_DIR"] = os.path.join(CACHE_DIR, "pip")
os.environ["TMPDIR"] = os.path.join(CACHE_DIR, "tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

# Resume ckpt → 로컬, best model → Drive (checkpoints/ symlink)
os.environ["DACON_LOCAL_CKPT"] = LOCAL_CKPT

import torch
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({vram:.1f} GB)")
else:
    print("GPU 없음! 런타임 > 런타임 유형 변경 > GPU 선택")

free = shutil.disk_usage('/content').free / 1e9
print(f"로컬 디스크 여유: {free:.1f} GB")
print(f"Resume ckpt 경로: {LOCAL_CKPT} (로컬, 세션 종료 시 삭제)")
print(f"Best model 경로: {DRIVE_DIR}/checkpoints/ (Drive, 영구)")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm>=1.0.20", "albumentations>=1.4.0",
    "opencv-python-headless", "scikit-learn", "pandas", "tqdm",
])
print("패키지 설치 완료")

In [ ]:
# 2. 소스 파일 복사 + 체크포인트 설정 (로컬 저장 → Drive 동기화)
import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/dacon"
WORK_DIR  = "/content/dacon"
DRIVE_CKPT = os.path.join(DRIVE_DIR, "checkpoints")
LOCAL_CKPT_BEST = os.path.join(WORK_DIR, "checkpoints")   # best model 로컬 저장
DRIVE_SUB  = os.path.join(DRIVE_DIR, "submissions")

# --- 소스 파일 복사 ---
SRC_FILES = ["models.py", "datasets.py", "train.py", "inference.py"]
for f in SRC_FILES:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join(WORK_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  [OK] {f}")
    else:
        print(f"  [MISSING] {f} — Drive에 업로드 필요: {DRIVE_DIR}/")

# --- 체크포인트: 로컬 디렉토리 (Drive symlink 안 함 = 빠른 저장) ---
os.makedirs(LOCAL_CKPT_BEST, exist_ok=True)

# --- submissions: Drive 직접 (작은 CSV라 느려도 OK) ---
os.makedirs(DRIVE_SUB, exist_ok=True)
sub_link = os.path.join(WORK_DIR, "submissions")
if os.path.islink(sub_link):
    os.unlink(sub_link)
elif os.path.isdir(sub_link):
    shutil.rmtree(sub_link)
os.symlink(DRIVE_SUB, sub_link)
print(f"  submissions/ → Drive (영구)")

# --- Drive에 기존 best 가중치 있으면 로컬로 복사 (resume용) ---
os.makedirs(DRIVE_CKPT, exist_ok=True)
copied = 0
for f in sorted(os.listdir(DRIVE_CKPT)):
    if not f.endswith('.pth'):
        continue
    src = os.path.join(DRIVE_CKPT, f)
    dst = os.path.join(LOCAL_CKPT_BEST, f)
    if not os.path.exists(dst):
        mb = os.path.getsize(src) / 1e6
        print(f"  ← Drive→로컬: {f} ({mb:.0f} MB)")
        shutil.copy2(src, dst)
        copied += 1
if copied == 0:
    print("  Drive 체크포인트 없음 또는 이미 로컬에 존재")

# --- 동기화 헬퍼 함수 (학습 후 호출) ---
def sync_best_to_drive(backbone=None):
    """로컬 best 가중치 → Drive 복사 (ckpt 제외, best만)"""
    count = 0
    for f in sorted(os.listdir(LOCAL_CKPT_BEST)):
        if not f.endswith('.pth'):
            continue
        if '_ckpt' in f:
            continue  # resume ckpt는 로컬에만
        if backbone and not f.startswith(backbone):
            continue
        src = os.path.join(LOCAL_CKPT_BEST, f)
        dst = os.path.join(DRIVE_CKPT, f)
        src_size = os.path.getsize(src)
        # 이미 동일 크기면 스킵
        if os.path.exists(dst) and os.path.getsize(dst) == src_size:
            continue
        mb = src_size / 1e6
        print(f"  → Drive: {f} ({mb:.0f} MB)")
        shutil.copy2(src, dst)
        count += 1
    if count == 0:
        print("  동기화할 새 가중치 없음")
    else:
        print(f"  {count}개 가중치 Drive 동기화 완료")

print(f"\n체크포인트 저장: 로컬 ({LOCAL_CKPT_BEST})")
print("학습 후 sync_best_to_drive() 호출하면 Drive에 복사됨")
print(f"Drive 체크포인트: {DRIVE_CKPT}")

In [ ]:
# 2.5 코랩 경쟁 모드 패치 (로컬 파일 영향 없음)
# 더 강한 증강 + CenterCrop TTA를 코랩 런타임에만 적용

from pathlib import Path

path = Path("/content/dacon/datasets.py")
text = path.read_text(encoding="utf-8")

old_train = '''def get_train_transforms(img_size=384):
    \"\"\"경쟁급 최대 증강\"\"\"
    return A.Compose([
        A.RandomResizedCrop(
            (img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.05, 0.05), scale=(0.93, 1.07),
            rotate=(-10, 10), p=0.5),
        A.Perspective(scale=(0.02, 0.06), p=0.3),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.HueSaturationValue(
                hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=1.0),
            A.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=1.0),
        ], p=0.8),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
        ], p=0.2),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.10), p=1.0),
            A.ISONoise(p=1.0),
        ], p=0.2),
        A.RandomShadow(p=0.15),
        A.RandomFog(fog_coef_range=(0.1, 0.35), p=0.1),
        A.CoarseDropout(
            num_holes_range=(1, 6),
            hole_height_range=(16, 40), hole_width_range=(16, 40), p=0.25),
        A.ToGray(p=0.05),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])'''

new_train = '''def get_train_transforms(img_size=384):
    \"\"\"코랩 경쟁 모드용 강화 증강\"\"\"
    return A.Compose([
        A.RandomResizedCrop(
            (img_size, img_size), scale=(0.72, 1.0), ratio=(0.88, 1.12), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.08, 0.08), scale=(0.90, 1.10),
            rotate=(-12, 12), p=0.6),
        A.Perspective(scale=(0.03, 0.08), p=0.35),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.25, contrast_limit=0.25, p=1.0),
            A.HueSaturationValue(
                hue_shift_limit=12, sat_shift_limit=18, val_shift_limit=18, p=1.0),
            A.ColorJitter(
                brightness=0.25, contrast=0.25, saturation=0.20, hue=0.06, p=1.0),
            A.CLAHE(clip_limit=(1, 4), p=1.0),
        ], p=0.9),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.1), p=1.0),
        ], p=0.25),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.12), p=1.0),
            A.ISONoise(p=1.0),
            A.ImageCompression(quality_range=(45, 90), p=1.0),
        ], p=0.25),
        A.RandomShadow(p=0.20),
        A.RandomFog(fog_coef_range=(0.1, 0.35), p=0.12),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(12, 56), hole_width_range=(12, 56), p=0.35),
        A.ToGray(p=0.08),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])'''

old_tta = '''def get_tta_transforms(img_size=384):
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    crop = int(img_size * 0.9)
    return [
        # 0: base (center crop)
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        # 1: hflip
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        # 2: brightness
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        # 3: tighter crop (90% of 90% = 81%)
        A.Compose([A.CenterCrop(int(img_size * 0.9), int(img_size * 0.9)),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
    ]'''

new_tta = '''def get_tta_transforms(img_size=384):
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    crop = int(img_size * 0.9)
    return [
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.CLAHE(clip_limit=(1, 3), p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.Affine(scale=(0.97, 1.03), translate_percent=(-0.02, 0.02), rotate=(-3, 3), p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(int(img_size * 0.9), int(img_size * 0.9)),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
    ]'''

if old_train in text:
    text = text.replace(old_train, new_train)
else:
    print('[WARN] train transform block not matched')

if old_tta in text:
    text = text.replace(old_tta, new_tta)
else:
    print('[WARN] tta transform block not matched')

path.write_text(text, encoding='utf-8')
print('코랩 런타임용 datasets.py 패치 완료')
print('  - train augmentation 강화')
print('  - TTA 6개 (전부 CenterCrop 90% 적용)')

In [ ]:
# 3. 데이터 압축 해제 (Drive data.zip → 로컬)
# Drive에서 폴더 복사는 너무 느림 → zip 해제가 훨씬 빠름

import os, shutil, zipfile, time

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")
ZIP_PATH   = os.path.join(DRIVE_DIR, "data.zip")

# 기존 data/ 정리
if os.path.islink(LOCAL_DATA):
    os.unlink(LOCAL_DATA)
elif os.path.isdir(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
os.makedirs(LOCAL_DATA, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print(f"[ERROR] {ZIP_PATH} 를 찾을 수 없습니다.")
    print(f"  Drive에 data.zip을 업로드하세요: {DRIVE_DIR}/data.zip")
    print(f"  zip 안에 open/, shapestacks/ 폴더가 있어야 합니다.")
else:
    zip_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f"data.zip 발견: {zip_mb:.0f} MB")
    print(f"해제 중... (로컬 SSD → 빠름)")

    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA)
    elapsed = time.time() - t0
    print(f"  해제 완료: {elapsed:.1f}초")

# --- 검증 ---
for f in ["open/train.csv", "open/dev.csv", "open/sample_submission.csv"]:
    p = os.path.join(LOCAL_DATA, f)
    ok = "[OK]" if os.path.exists(p) else "[FAIL]"
    print(f"  {ok} {f}")

for split in ["train", "dev", "test"]:
    d = os.path.join(LOCAL_DATA, "open", split)
    if os.path.isdir(d):
        n = len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])
        print(f"  {split}/: {n}개")

ss = os.path.join(LOCAL_DATA, "shapestacks", "shapestacks", "recordings")
if os.path.exists(ss):
    h6 = [d for d in os.listdir(ss) if "h=6" in d]
    print(f"  ShapeStacks h=6: {len(h6)} scenarios")
else:
    print("  [SKIP] ShapeStacks 없음 (Dacon only)")

free = shutil.disk_usage('/content').free / 1e9
print(f"\n로컬 디스크 여유: {free:.1f} GB")

In [ ]:
# 5. 설정 검증
import os
import shutil

WORK_DIR  = "/content/dacon"
DRIVE_DIR = "/content/drive/MyDrive/dacon"

print("=" * 50)
print("  환경 검증")
print("=" * 50)

for f in ["models.py", "datasets.py", "train.py", "inference.py"]:
    ok = os.path.exists(os.path.join(WORK_DIR, f))
    print(f"  {'[OK]' if ok else '[FAIL]'} {f}")

tc = os.path.join(WORK_DIR, "data", "open", "train.csv")
print(f"  {'[OK]' if os.path.exists(tc) else '[FAIL]'} train.csv")

dc = os.path.join(WORK_DIR, "data", "open", "dev.csv")
print(f"  {'[OK]' if os.path.exists(dc) else '[FAIL]'} dev.csv")

ss = os.path.join(WORK_DIR, "data", "shapestacks", "shapestacks", "recordings")
if os.path.exists(ss):
    h6 = [d for d in os.listdir(ss) if "h=6" in d]
    print(f"  [OK] ShapeStacks h=6: {len(h6)} scenarios")
else:
    print(f"  [SKIP] ShapeStacks 없음 (Dacon only)")

# 체크포인트: 로컬 디렉토리 (Drive symlink 아님)
ckpt_dir = os.path.join(WORK_DIR, "checkpoints")
is_local = os.path.isdir(ckpt_dir) and not os.path.islink(ckpt_dir)
print(f"  {'[OK]' if is_local else '[WARN]'} checkpoints/ (로컬 저장)")

# submissions: Drive 심볼링크
sub_link = os.path.join(WORK_DIR, "submissions")
is_drive = os.path.islink(sub_link) and "drive" in os.readlink(sub_link)
print(f"  {'[OK]' if is_drive else '[FAIL]'} submissions/ → Drive")

# 로컬 체크포인트 현황
ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pth')] if os.path.isdir(ckpt_dir) else []
resume_ckpts = [c for c in ckpts if "ckpt" in c]
best_ckpts   = [c for c in ckpts if "ckpt" not in c]

print()
if best_ckpts:
    print(f"  로컬 Best 가중치: {len(best_ckpts)}개")
    for c in sorted(best_ckpts):
        mb = os.path.getsize(os.path.join(ckpt_dir, c)) / 1e6
        print(f"    {c} ({mb:.0f} MB)")
if resume_ckpts:
    print(f"  로컬 Resume ckpt: {len(resume_ckpts)}개")
    for c in sorted(resume_ckpts):
        print(f"    {c}")
if not ckpts:
    print("  체크포인트 없음 → 새로 학습 시작")

# Drive 체크포인트 현황
drive_ckpt = os.path.join(DRIVE_DIR, "checkpoints")
drive_ckpts = [f for f in os.listdir(drive_ckpt) if f.endswith('.pth')] if os.path.isdir(drive_ckpt) else []
if drive_ckpts:
    print(f"\n  Drive 백업: {len(drive_ckpts)}개")
    for c in sorted(drive_ckpts):
        mb = os.path.getsize(os.path.join(drive_ckpt, c)) / 1e6
        print(f"    {c} ({mb:.0f} MB)")

free = shutil.disk_usage('/content').free / 1e9
print(f"\n  디스크 여유: {free:.1f} GB")
print("=" * 50)

In [ ]:
# 6. 학습 설정

BACKBONE = "eva_giant"
PRETRAIN_EPOCHS = 0
FINETUNE_EPOCHS = 30
PATIENCE = 15

MERGE_DEV = True
LOSS = "ce"
NO_MIXUP = True
HEAD_TYPE = "simple"
SIMPLE_AUG = True
SCHEDULER = "cosine_wr"
HEAD_LR = 2e-4
BB_LR = 2e-5
WEIGHT_DECAY = 0.01
DROP_RATE = 0.3

USE_VIDEO = False

import os
os.chdir("/content/dacon")

print(f"백본: {BACKBONE}")
print(f"Pretrain: {PRETRAIN_EPOCHS} ep -> Finetune: {FINETUNE_EPOCHS} ep")
print(f"Patience: {PATIENCE}")
print(f"  merge_dev={MERGE_DEV} | loss={LOSS} | no_mixup={NO_MIXUP}")
print(f"  head={HEAD_TYPE} | aug={'simple' if SIMPLE_AUG else 'full'} | sched={SCHEDULER}")
print(f"  head_lr={HEAD_LR} | bb_lr={BB_LR} | wd={WEIGHT_DECAY} | drop={DROP_RATE}")

pt_path = os.path.join("checkpoints", f"{BACKBONE}_pretrained.pth")
if os.path.exists(pt_path):
    mb = os.path.getsize(pt_path) / 1e6
    print(f"\n[INFO] 사전학습 체크포인트: {pt_path} ({mb:.0f} MB)")

for fold in range(5):
    best = os.path.join("checkpoints", f"{BACKBONE}_fold{fold}.pth")
    status = "완료" if os.path.exists(best) else "미완료"
    print(f"  Fold {fold}: {status}")

In [ ]:
# 7. Stage 1: Pretrain (ShapeStacks h=6 + Dacon)
# 경쟁 모드에서는 기본적으로 건너뜀

import os, sys, shlex, subprocess
os.chdir("/content/dacon")

if PRETRAIN_EPOCHS <= 0:
    print("경쟁 모드 기본값: pretrain 생략")
else:
    cmd = [
        sys.executable, "-u", "train.py",
        "--backbone", BACKBONE,
        "--stage", "pretrain",
        "--pretrain_epochs", str(PRETRAIN_EPOCHS),
        "--batch_size_override", "16",
        "--grad_checkpointing",
        "--resume",
        "--num_workers", "0",
    ]
    cmd_str = shlex.join(cmd)
    print(f"CMD: {cmd_str}\n" + "="*50)

    my_env = os.environ.copy()
    my_env["TQDM_FORCE_TTY"] = "1"

    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=my_env
    )

    while True:
        char = process.stdout.read(1)
        if not char and process.poll() is not None:
            break
        sys.stdout.write(char)
        sys.stdout.flush()

    process.wait()
    if process.returncode != 0:
        print(f"\n[ERROR] Pretrain 실패 (exit code {process.returncode})")

In [ ]:
# 8-0. Finetune Fold 0

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 0
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--grad_checkpointing",
    "--resume",
    "--skip_completed",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)

In [ ]:
# 8-1. Finetune Fold 1

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 1
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--grad_checkpointing",
    "--resume",
    "--skip_completed",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)

In [ ]:
# 8-2. Finetune Fold 2

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 2
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--grad_checkpointing",
    "--resume",
    "--skip_completed",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)

In [ ]:
# 8-3. Finetune Fold 3

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 3
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--grad_checkpointing",
    "--resume",
    "--skip_completed",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)

In [ ]:
# 8-4. Finetune Fold 4

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 4
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--grad_checkpointing",
    "--resume",
    "--skip_completed",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)

In [ ]:
# 9. 추론 + 제출 파일 생성

import os, sys, shlex, subprocess, glob
import pandas as pd
os.chdir("/content/dacon")

cmd = [
    sys.executable, "-u", "inference.py",
    "--backbones", BACKBONE,
    "--head_type", HEAD_TYPE,
    "--tta",
    "--temperature", "1.0",
    "--num_workers", "0",
]
cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()

if process.returncode != 0:
    print(f"\n[ERROR] 추론 실패 (exit code {process.returncode})")
else:
    subs = sorted(glob.glob("/content/dacon/submissions/*.csv"))
    if subs:
        latest = subs[-1]
        df = pd.read_csv(latest)
        print(f"\n최신 제출: {os.path.basename(latest)}")
        print(f"  샘플 수: {len(df)}")
        print(f"  unstable_prob: mean={df['unstable_prob'].mean():.4f}, std={df['unstable_prob'].std():.4f}")
        print(df.head())
        print(f"\n제출 파일은 Drive에 저장됨: /content/drive/MyDrive/dacon/submissions/")
    else:
        print("\n제출 파일이 생성되지 않았습니다. 체크포인트가 있는지 확인하세요.")

In [ ]:
# 10. Dev 검증 (단일 백본)

import os, sys, shlex, subprocess
os.chdir("/content/dacon")

cmd = [
    sys.executable, "-u", "inference.py",
    "--backbones", BACKBONE,
    "--head_type", HEAD_TYPE,
    "--tta",
    "--validate",
    "--num_workers", "0",
]
cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] 검증 실패 (exit code {process.returncode})")

In [ ]:
# 11. Test 제출 파일 검증
import pandas as pd

df = pd.read_csv('submission_eva_giant_3seeds_tta.csv')

unstable_mean = df['unstable_prob'].mean()
unstable_std = df['unstable_prob'].std()

print(f"unstable_prob: mean={unstable_mean:.4f}, std={unstable_std:.4f}")
print(df[['id', 'unstable_prob', 'stable_prob']].head().to_string())